# Huấn luyện DeepVQE Phase 2 Mamba trên Kaggle

Notebook này dùng cấu hình `Mamba_b2_h384` để thay bottleneck GRU bằng Native PyTorch Mamba/SSM có cache streaming. Notebook cũng có sẵn cell smoke test, RTF benchmark, quality eval và ONNX export.


## 1. Cài đặt môi trường & chuẩn bị Kaggle Working Dir


In [ ]:
!pip install -q wandb gdown matplotlib soundfile pandas tqdm einops pesq pystoi pyyaml torchmetrics psutil onnx onnxruntime onnxsim

import os
import sys
import shutil
from pathlib import Path

WORK_DIR = Path('/kaggle/working/DeepVQE_Workspace')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Working dir: {Path.cwd()}')

SHARED_CHECKPOINTS_DIR = Path('/kaggle/working/checkpoint_phase2_mamba')
SHARED_CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Checkpoint dir: {SHARED_CHECKPOINTS_DIR}')


## 2. Clone mã nguồn DeepVQE


In [ ]:
import subprocess

GIT_REPO = 'https://github.com/hoxuanphu/Pdeepvqe.git'
GIT_BRANCH = None  # Ví dụ: 'phase2-mamba' nếu bạn đã push branch riêng.
REPO_DIR = WORK_DIR / 'deepvqe'

if not REPO_DIR.exists():
    cmd = ['git', 'clone']
    if GIT_BRANCH:
        cmd += ['--branch', GIT_BRANCH]
    cmd += [GIT_REPO, str(REPO_DIR)]
    subprocess.run(cmd, check=True)
else:
    print(f'Thư mục {REPO_DIR} đã tồn tại. Đang cập nhật code mới nhất...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f'Repo dir: {Path.cwd()}')


## 2.5 Áp dụng code Phase 2 Mamba vào runtime


In [ ]:
from pathlib import Path
import importlib

PHASE2_FILES = {
  "ablation/modules/__init__.py": "\"\"\"Reusable modules for DeepVQE ablation variants.\"\"\"\n\n",
  "ablation/modules/mamba.py": "\"\"\"Native PyTorch Mamba-style state space blocks.\n\nThis implementation favors a small, explicit streaming state contract over\ncustom CUDA kernels so ablation variants remain portable to ONNX/runtime\ndeployments.\n\"\"\"\n\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass NativeMambaBlock(nn.Module):\n    \"\"\"Causal Mamba-style block with an explicit recurrent cache.\n\n    Input and output are shaped ``(B, T, D)``. The streaming cache is a tuple\n    ``(conv_cache, ssm_state)`` where ``conv_cache`` stores the previous\n    ``d_conv - 1`` projected samples and ``ssm_state`` stores per-channel SSM\n    state ``(B, d_inner, d_state)``.\n    \"\"\"\n\n    def __init__(\n        self,\n        d_model,\n        d_state=16,\n        d_conv=4,\n        expand=2,\n        dt_rank=\"auto\",\n        dt_min=0.001,\n        dt_max=0.1,\n        dt_init_floor=1e-4,\n    ):\n        super().__init__()\n        self.d_model = int(d_model)\n        self.d_state = int(d_state)\n        self.d_conv = int(d_conv)\n        self.expand = int(expand)\n        self.d_inner = self.d_model * self.expand\n        if self.d_state < 1:\n            raise ValueError(\"d_state must be >= 1\")\n        if self.d_conv < 1:\n            raise ValueError(\"d_conv must be >= 1\")\n        if dt_rank == \"auto\":\n            dt_rank = math.ceil(self.d_model / 16)\n        self.dt_rank = int(dt_rank)\n\n        self.in_proj = nn.Linear(self.d_model, self.d_inner * 2, bias=False)\n        self.conv1d = nn.Conv1d(\n            self.d_inner,\n            self.d_inner,\n            kernel_size=self.d_conv,\n            groups=self.d_inner,\n            bias=True,\n        )\n        self.activation = nn.SiLU()\n        self.x_proj = nn.Linear(self.d_inner, self.dt_rank + 2 * self.d_state, bias=False)\n        self.dt_proj = nn.Linear(self.dt_rank, self.d_inner, bias=True)\n        self.out_proj = nn.Linear(self.d_inner, self.d_model, bias=False)\n\n        a = torch.arange(1, self.d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)\n        self.A_log = nn.Parameter(torch.log(a))\n        self.D = nn.Parameter(torch.ones(self.d_inner))\n\n        dt = torch.exp(\n            torch.rand(self.d_inner) * (math.log(dt_max) - math.log(dt_min))\n            + math.log(dt_min)\n        ).clamp(min=dt_init_floor)\n        inv_dt = dt + torch.log(-torch.expm1(-dt))\n        with torch.no_grad():\n            self.dt_proj.bias.copy_(inv_dt)\n\n    def init_cache(self, batch_size, device=None, dtype=None):\n        conv_len = max(0, self.d_conv - 1)\n        conv_cache = torch.zeros(\n            int(batch_size),\n            self.d_inner,\n            conv_len,\n            device=device,\n            dtype=dtype,\n        )\n        ssm_state = torch.zeros(\n            int(batch_size),\n            self.d_inner,\n            self.d_state,\n            device=device,\n            dtype=dtype,\n        )\n        return conv_cache, ssm_state\n\n    def _ssm_step(self, x, ssm_state):\n        x_proj = self.x_proj(x)\n        dt, b, c = torch.split(\n            x_proj,\n            [self.dt_rank, self.d_state, self.d_state],\n            dim=-1,\n        )\n        dt = F.softplus(self.dt_proj(dt))\n        a = -torch.exp(self.A_log.float()).to(dtype=x.dtype, device=x.device)\n        d_a = torch.exp(dt.unsqueeze(-1) * a.unsqueeze(0))\n        d_b = dt.unsqueeze(-1) * b.unsqueeze(1)\n        ssm_state = ssm_state * d_a + x.unsqueeze(-1) * d_b\n        y = torch.sum(ssm_state * c.unsqueeze(1), dim=-1)\n        y = y + x * self.D.to(dtype=x.dtype, device=x.device)\n        return y, ssm_state\n\n    def _causal_conv(self, x, conv_cache=None):\n        x_t = x.transpose(1, 2)\n        if conv_cache is None:\n            x_t = F.pad(x_t, (self.d_conv - 1, 0))\n        else:\n            x_t = torch.cat([conv_cache, x_t], dim=2)\n        x_t = self.conv1d(x_t)\n        return x_t.transpose(1, 2)\n\n    def forward(self, x, cache=None):\n        if x.shape[-1] != self.d_model:\n            raise ValueError(f\"NativeMambaBlock expected {self.d_model} features, got {x.shape[-1]}\")\n\n        xz = self.in_proj(x)\n        x_part, z = xz.chunk(2, dim=-1)\n        conv_cache_in = None if cache is None else cache[0]\n        projected = x_part.transpose(1, 2)\n        x_part = self.activation(self._causal_conv(x_part, conv_cache_in))\n\n        if cache is None:\n            ssm_state = torch.zeros(\n                x.shape[0],\n                self.d_inner,\n                self.d_state,\n                device=x.device,\n                dtype=x.dtype,\n            )\n        else:\n            _, ssm_state = cache\n\n        outputs = []\n        for t in range(x_part.shape[1]):\n            y_t, ssm_state = self._ssm_step(x_part[:, t], ssm_state)\n            outputs.append(y_t)\n        y = torch.stack(outputs, dim=1)\n        y = y * self.activation(z)\n        y = self.out_proj(y)\n\n        conv_len = max(0, self.d_conv - 1)\n        if conv_len > 0:\n            if conv_cache_in is not None:\n                projected = torch.cat([conv_cache_in, projected], dim=2)\n            conv_cache = projected[:, :, -conv_len:].contiguous()\n        else:\n            conv_cache = x.new_zeros(x.shape[0], self.d_inner, 0)\n        return y, (conv_cache, ssm_state)\n\n    def step(self, x, cache):\n        \"\"\"Run one frame shaped ``(B, D)`` and return ``(B, D), cache``.\"\"\"\n        if x.shape[-1] != self.d_model:\n            raise ValueError(f\"NativeMambaBlock expected {self.d_model} features, got {x.shape[-1]}\")\n        conv_cache, ssm_state = cache\n        xz = self.in_proj(x)\n        x_part, z = xz.chunk(2, dim=-1)\n        conv_input = torch.cat([conv_cache, x_part.unsqueeze(-1)], dim=2)\n        weight = self.conv1d.weight.squeeze(1)\n        x_conv = torch.sum(conv_input * weight.unsqueeze(0), dim=2)\n        if self.conv1d.bias is not None:\n            x_conv = x_conv + self.conv1d.bias\n        x_conv = self.activation(x_conv)\n        conv_len = max(0, self.d_conv - 1)\n        if conv_len > 0:\n            conv_cache = conv_input[:, :, -conv_len:].contiguous()\n        else:\n            conv_cache = conv_input[:, :, :0].contiguous()\n\n        y, ssm_state = self._ssm_step(x_conv, ssm_state)\n        y = y * self.activation(z)\n        y = self.out_proj(y)\n        return y, (conv_cache, ssm_state)\n\n\nclass ResidualMambaBlock(nn.Module):\n    \"\"\"LayerNorm + Mamba block with residual output.\"\"\"\n\n    def __init__(self, d_model, **mamba_kwargs):\n        super().__init__()\n        self.norm = nn.LayerNorm(int(d_model))\n        self.mamba = NativeMambaBlock(d_model, **mamba_kwargs)\n\n    @property\n    def d_inner(self):\n        return self.mamba.d_inner\n\n    @property\n    def d_state(self):\n        return self.mamba.d_state\n\n    @property\n    def d_conv(self):\n        return self.mamba.d_conv\n\n    def init_cache(self, batch_size, device=None, dtype=None):\n        return self.mamba.init_cache(batch_size, device=device, dtype=dtype)\n\n    def forward(self, x, cache=None):\n        y, cache = self.mamba(self.norm(x), cache)\n        return x + y, cache\n\n    def step(self, x, cache):\n        y, cache = self.mamba.step(self.norm(x), cache)\n        return x + y, cache\n\n\nclass MambaStack(nn.Module):\n    \"\"\"Stack of residual Mamba blocks with flat cache helpers.\"\"\"\n\n    def __init__(\n        self,\n        d_model,\n        num_blocks=2,\n        hidden_size=None,\n        d_state=16,\n        d_conv=4,\n        expand=2,\n        dt_rank=\"auto\",\n    ):\n        super().__init__()\n        self.d_model = int(d_model)\n        self.num_blocks = int(num_blocks)\n        self.hidden_size = int(hidden_size) if hidden_size is not None else self.d_model\n        if self.num_blocks < 1:\n            raise ValueError(\"num_blocks must be >= 1\")\n\n        self.in_proj = (\n            nn.Linear(self.d_model, self.hidden_size)\n            if self.hidden_size != self.d_model\n            else nn.Identity()\n        )\n        self.blocks = nn.ModuleList(\n            [\n                ResidualMambaBlock(\n                    self.hidden_size,\n                    d_state=d_state,\n                    d_conv=d_conv,\n                    expand=expand,\n                    dt_rank=dt_rank,\n                )\n                for _ in range(self.num_blocks)\n            ]\n        )\n        self.norm = nn.LayerNorm(self.hidden_size)\n        self.out_proj = (\n            nn.Linear(self.hidden_size, self.d_model)\n            if self.hidden_size != self.d_model\n            else nn.Identity()\n        )\n\n    def cache_names(self, prefix=\"mamba\"):\n        names = []\n        for idx in range(self.num_blocks):\n            names.append(f\"{prefix}{idx + 1}_conv_cache\")\n            names.append(f\"{prefix}{idx + 1}_ssm_state\")\n        return tuple(names)\n\n    def init_cache(self, batch_size, device=None, dtype=None):\n        cache = []\n        for block in self.blocks:\n            block_cache = block.init_cache(batch_size, device=device, dtype=dtype)\n            cache.extend(block_cache)\n        return tuple(cache)\n\n    def _pack_cache(self, flat_cache):\n        if len(flat_cache) != self.num_blocks * 2:\n            raise ValueError(f\"Expected {self.num_blocks * 2} Mamba cache tensors, got {len(flat_cache)}\")\n        return [\n            (flat_cache[2 * idx], flat_cache[2 * idx + 1])\n            for idx in range(self.num_blocks)\n        ]\n\n    def _unpack_cache(self, block_caches):\n        flat = []\n        for conv_cache, ssm_state in block_caches:\n            flat.extend([conv_cache, ssm_state])\n        return tuple(flat)\n\n    def forward(self, x, cache=None):\n        y = self.in_proj(x)\n        block_caches = (\n            [None] * self.num_blocks\n            if cache is None\n            else self._pack_cache(cache)\n        )\n        new_caches = []\n        for block, block_cache in zip(self.blocks, block_caches):\n            y, block_cache = block(y, block_cache)\n            new_caches.append(block_cache)\n        y = self.out_proj(self.norm(y))\n        return y, self._unpack_cache(new_caches)\n\n    def step(self, x, cache):\n        y = self.in_proj(x)\n        new_caches = []\n        for block, block_cache in zip(self.blocks, self._pack_cache(cache)):\n            y, block_cache = block.step(y, block_cache)\n            new_caches.append(block_cache)\n        y = self.out_proj(self.norm(y))\n        return y, self._unpack_cache(new_caches)\n",
  "ablation/deepvqe_ablation.py": "\"\"\"Parameterized DeepVQE ablation variants.\n\nThis module intentionally leaves ``deepvqe.py`` untouched. The Baseline\nconfiguration is state-dict compatible with ``deepvqe.DeepVQE`` and can load\nbaseline weights with ``strict=True``. Non-baseline variants change module\nstructure and should load baseline weights with ``strict=False`` only.\n\"\"\"\n\nfrom copy import deepcopy\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom einops import rearrange\n\nfrom deepvqe import CCM, FE, SubpixelConv2d\nfrom ablation.modules.mamba import MambaStack\nfrom stream.modules.convolution import StreamConv2d\n\n\nBASE_GRU_HIDDEN = 64 * 9\n\n\nABLATION_CONFIGS = {\n    \"Baseline\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    \"D1a_gru704\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": 704,\n    },\n    \"D1b_gru768\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": 768,\n    },\n    \"D2_temporal_refine\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n        \"temporal_refine\": True,\n        \"temporal_refine_kernel\": 3,\n    },\n    \"Mamba_b2_h384\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n        \"sequence_model\": \"mamba\",\n        \"mamba_blocks\": 2,\n        \"mamba_hidden\": 384,\n        \"mamba_state\": 16,\n        \"mamba_conv\": 4,\n        \"mamba_expand\": 2,\n    },\n    \"B1a\": {\n        \"prelu_type\": \"shared\",\n        \"dw_residual\": False,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    \"B1b\": {\n        \"prelu_type\": \"per_channel\",\n        \"dw_residual\": False,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    \"B2\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"use_eca_f\": True,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    \"B3a\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": \"eca_f\",\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    \"B3b\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": \"se_f\",\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    \"C1\": {\n        \"prelu_type\": None,\n        \"dw_residual\": True,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    \"C2a\": {\n        \"prelu_type\": None,\n        \"dw_residual\": True,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": True,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    \"C2b\": {\n        \"prelu_type\": None,\n        \"dw_residual\": True,\n        \"use_eca_f\": True,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    # C3/C4 depend on the B1 tie-breaker. Per-channel matches the previous\n    # local default and can be overridden to \"shared\" once B1a wins.\n    \"C3\": {\n        \"prelu_type\": \"per_channel\",\n        \"dw_residual\": True,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    \"C4\": {\n        \"prelu_type\": \"per_channel\",\n        \"dw_residual\": True,\n        \"use_eca_f\": True,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n}\n\n\nLEGACY_CONFIG_ALIASES = {\n    \"C1b\": \"C1\",\n    \"C2\": \"C2b\",\n}\n\n\nLEGACY_CONFIGS = {\n    \"C1a-g2\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"res_groups\": 2,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    \"C1a-g4\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"res_groups\": 4,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n    # Legacy B3 was ECA-F in main encoder/decoder blocks. Roadmap B3a/B3b\n    # now cover skip gating, but this remains loadable for older runs.\n    \"B3\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"use_eca_f\": True,\n        \"main_block_eca_f\": True,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": BASE_GRU_HIDDEN,\n    },\n}\n\n\ndef _activation_to_prelu_type(activation):\n    if activation in (None, \"elu\"):\n        return None\n    if activation == \"prelu_shared\":\n        return \"shared\"\n    if activation == \"prelu_channel\":\n        return \"per_channel\"\n    if activation in (\"shared\", \"per_channel\"):\n        return activation\n    raise ValueError(\n        f\"Unsupported activation {activation!r}; expected 'elu', \"\n        \"'prelu_shared', or 'prelu_channel'.\"\n    )\n\n\ndef _normalize_model_config(config):\n    \"\"\"Accept planned config keys plus legacy local keys.\"\"\"\n    cfg = deepcopy(config)\n\n    if \"activation\" in cfg:\n        activation = cfg.pop(\"activation\")\n        legacy_prelu_type = _activation_to_prelu_type(activation)\n        if legacy_prelu_type is not None or cfg.get(\"prelu_type\") is None:\n            cfg[\"prelu_type\"] = legacy_prelu_type\n\n    if \"attn_type\" in cfg:\n        attn_type = cfg.pop(\"attn_type\")\n        if attn_type in (None, \"none\"):\n            cfg[\"use_eca_f\"] = bool(cfg.get(\"use_eca_f\", False))\n        elif attn_type == \"eca_f\":\n            cfg[\"use_eca_f\"] = True\n        else:\n            raise ValueError(\n                \"Only attn_type='eca_f' is supported in Phase 1 because \"\n                \"ECA-CT has no streaming cache contract.\"\n            )\n\n    if \"res_conv_type\" in cfg:\n        res_conv_type = cfg.pop(\"res_conv_type\")\n        if res_conv_type == \"standard\":\n            cfg[\"dw_residual\"] = bool(cfg.get(\"dw_residual\", False))\n        elif res_conv_type == \"dw_separable\":\n            cfg[\"dw_residual\"] = True\n        elif res_conv_type == \"grouped\":\n            cfg[\"dw_residual\"] = False\n        else:\n            raise ValueError(\n                f\"Unsupported res_conv_type {res_conv_type!r}; expected \"\n                \"'standard', 'grouped', or 'dw_separable'.\"\n            )\n\n    cfg.setdefault(\"prelu_type\", None)\n    cfg.setdefault(\"dw_residual\", False)\n    cfg.setdefault(\"use_eca_f\", False)\n    cfg.setdefault(\"main_block_eca_f\", False)\n    cfg.setdefault(\"gru_hidden\", BASE_GRU_HIDDEN)\n    cfg.setdefault(\"res_groups\", None)\n    cfg.setdefault(\"skip_gate\", None)\n    cfg.setdefault(\"dw_subpixel\", False)\n    cfg.setdefault(\"temporal_refine\", False)\n    cfg.setdefault(\"temporal_refine_kernel\", 3)\n    cfg.setdefault(\"sequence_model\", \"gru\")\n    cfg.setdefault(\"mamba_blocks\", 2)\n    cfg.setdefault(\"mamba_hidden\", None)\n    cfg.setdefault(\"mamba_state\", 16)\n    cfg.setdefault(\"mamba_conv\", 4)\n    cfg.setdefault(\"mamba_expand\", 2)\n\n    if cfg[\"skip_gate\"] in (\"none\", \"identity\", False):\n        cfg[\"skip_gate\"] = None\n    if cfg[\"skip_gate\"] == \"se\":\n        cfg[\"skip_gate\"] = \"se_f\"\n    if cfg[\"skip_gate\"] not in (None, \"eca_f\", \"se_f\"):\n        raise ValueError(\"skip_gate must be None, 'eca_f', or 'se_f'\")\n\n    if cfg[\"sequence_model\"] in (None, \"rnn\"):\n        cfg[\"sequence_model\"] = \"gru\"\n    if cfg[\"sequence_model\"] not in (\"gru\", \"mamba\"):\n        raise ValueError(\"sequence_model must be 'gru' or 'mamba'\")\n    cfg[\"mamba_blocks\"] = int(cfg[\"mamba_blocks\"])\n    cfg[\"mamba_state\"] = int(cfg[\"mamba_state\"])\n    cfg[\"mamba_conv\"] = int(cfg[\"mamba_conv\"])\n    cfg[\"mamba_expand\"] = int(cfg[\"mamba_expand\"])\n    if cfg[\"mamba_hidden\"] is not None:\n        cfg[\"mamba_hidden\"] = int(cfg[\"mamba_hidden\"])\n    if cfg[\"sequence_model\"] != \"gru\" and cfg[\"temporal_refine\"]:\n        raise ValueError(\"temporal_refine is currently only supported with sequence_model='gru'\")\n\n    allowed = {\n        \"prelu_type\",\n        \"dw_residual\",\n        \"use_eca_f\",\n        \"main_block_eca_f\",\n        \"gru_hidden\",\n        \"res_groups\",\n        \"skip_gate\",\n        \"dw_subpixel\",\n        \"temporal_refine\",\n        \"temporal_refine_kernel\",\n        \"sequence_model\",\n        \"mamba_blocks\",\n        \"mamba_hidden\",\n        \"mamba_state\",\n        \"mamba_conv\",\n        \"mamba_expand\",\n    }\n    unknown = sorted(set(cfg) - allowed)\n    if unknown:\n        raise ValueError(f\"Unknown model config keys: {unknown}\")\n    return cfg\n\n\ndef get_ablation_config(config_id=\"Baseline\", **overrides):\n    \"\"\"Return a copy of a named ablation config with optional overrides.\"\"\"\n    if config_id in LEGACY_CONFIG_ALIASES:\n        config_id = LEGACY_CONFIG_ALIASES[config_id]\n\n    if config_id in ABLATION_CONFIGS:\n        config = deepcopy(ABLATION_CONFIGS[config_id])\n    elif config_id in LEGACY_CONFIGS:\n        config = deepcopy(LEGACY_CONFIGS[config_id])\n    else:\n        valid = \", \".join(list(ABLATION_CONFIGS) + list(LEGACY_CONFIG_ALIASES) + list(LEGACY_CONFIGS))\n        raise ValueError(f\"Unknown ablation config {config_id!r}. Valid configs: {valid}\")\n\n    config.update(overrides)\n    return _normalize_model_config(config)\n\n\ndef ActivationFactory(prelu_type=None, channels=None):\n    \"\"\"Build the configured activation for ``(B, C, T, F)`` feature tensors.\"\"\"\n    prelu_type = _activation_to_prelu_type(prelu_type)\n    if prelu_type is None:\n        return nn.ELU()\n    if prelu_type == \"shared\":\n        return nn.PReLU(num_parameters=1)\n    if prelu_type == \"per_channel\":\n        if channels is None:\n            raise ValueError(\"channels must be provided for prelu_type='per_channel'\")\n        return nn.PReLU(num_parameters=int(channels))\n    raise ValueError(\"prelu_type must be None, 'shared', or 'per_channel'\")\n\n\nclass CausalECA_F(nn.Module):\n    \"\"\"Frequency-pooled efficient channel attention.\n\n    Pooling is only over the frequency axis. The 1-D convolution slides across\n    channels for each frame independently, so no temporal cache is required.\n    \"\"\"\n\n    def __init__(self, channels, kernel_size=3):\n        super().__init__()\n        if kernel_size % 2 == 0:\n            raise ValueError(\"CausalECA_F kernel_size must be odd\")\n        self.channels = int(channels)\n        self.conv = nn.Conv1d(\n            1,\n            1,\n            kernel_size=kernel_size,\n            padding=(kernel_size - 1) // 2,\n            bias=False,\n        )\n        self.sigmoid = nn.Sigmoid()\n\n    def forward(self, x):\n        \"\"\"x: (B, C, T, F).\"\"\"\n        b, c, t, _ = x.shape\n        if c != self.channels:\n            raise ValueError(f\"CausalECA_F expected {self.channels} channels, got {c}\")\n\n        y = x.mean(dim=3, keepdim=True)  # (B, C, T, 1)\n        y = y.squeeze(3).permute(0, 2, 1).reshape(b * t, 1, c)\n        y = self.sigmoid(self.conv(y))\n        y = y.reshape(b, t, c).permute(0, 2, 1).unsqueeze(3)\n        return x * y\n\n\nECA_F = CausalECA_F\n\n\ndef _attention(channels, enabled):\n    return CausalECA_F(channels) if enabled else nn.Identity()\n\n\nclass FrequencySE(nn.Module):\n    \"\"\"Frequency-only squeeze/excitation for causal skip gating.\"\"\"\n\n    def __init__(self, channels, reduction=8):\n        super().__init__()\n        hidden = max(1, int(channels) // int(reduction))\n        self.gate = nn.Sequential(\n            nn.Conv2d(channels, hidden, kernel_size=1),\n            nn.ReLU(inplace=True),\n            nn.Conv2d(hidden, channels, kernel_size=1),\n            nn.Sigmoid(),\n        )\n\n    def forward(self, x):\n        y = x.mean(dim=3, keepdim=True)\n        return x * self.gate(y)\n\n\ndef _skip_gate(channels, gate_type):\n    if gate_type is None:\n        return nn.Identity()\n    if gate_type == \"eca_f\":\n        return CausalECA_F(channels)\n    if gate_type == \"se_f\":\n        return FrequencySE(channels)\n    raise ValueError(f\"Unsupported skip_gate={gate_type!r}\")\n\n\nclass CausalTemporalRefine(nn.Module):\n    \"\"\"Causal temporal residual refinement over flattened bottleneck features.\"\"\"\n\n    def __init__(self, features, kernel_size=3):\n        super().__init__()\n        features = int(features)\n        kernel_size = int(kernel_size)\n        if kernel_size < 1:\n            raise ValueError(\"temporal_refine_kernel must be >= 1\")\n        self.features = features\n        self.kernel_size = kernel_size\n        self.norm = nn.LayerNorm(features)\n        self.depthwise = nn.Conv1d(\n            features,\n            features,\n            kernel_size=kernel_size,\n            groups=features,\n            bias=False,\n        )\n        self.pointwise = nn.Conv1d(features, features, kernel_size=1)\n        self.elu = nn.ELU()\n\n    def forward(self, y):\n        \"\"\"y: (B, T, D). Returns residual delta with the same shape.\"\"\"\n        if y.shape[-1] != self.features:\n            raise ValueError(f\"CausalTemporalRefine expected {self.features} features, got {y.shape[-1]}\")\n        z = self.norm(y).transpose(1, 2)\n        z = F.pad(z, (self.kernel_size - 1, 0))\n        z = self.depthwise(z)\n        z = self.pointwise(z)\n        z = self.elu(z)\n        return z.transpose(1, 2)\n\n\nclass DWSubpixelConv2d(nn.Module):\n    \"\"\"Depthwise-separable version of the original SubpixelConv2d.\"\"\"\n\n    def __init__(self, in_channels, out_channels, kernel_size=(4, 3)):\n        super().__init__()\n        self.pad = nn.ZeroPad2d([1, 1, 3, 0])\n        self.depthwise = nn.Conv2d(in_channels, in_channels, kernel_size, groups=in_channels)\n        self.pointwise = nn.Conv2d(in_channels, out_channels * 2, kernel_size=1)\n\n    def forward(self, x):\n        y = self.pointwise(self.depthwise(self.pad(x)))\n        y = rearrange(y, \"b (r c) t f -> b c t (r f)\", r=2)\n        return y\n\n\nclass ResidualBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        channels,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        res_groups=None,\n    ):\n        super().__init__()\n        channels = int(channels)\n        self.pad = nn.ZeroPad2d([1, 1, 3, 0])\n        self.dw_residual = bool(dw_residual)\n        self.res_groups = res_groups\n\n        if self.dw_residual:\n            self.depthwise = nn.Conv2d(channels, channels, kernel_size=(4, 3), groups=channels)\n            self.pointwise = nn.Conv2d(channels, channels, kernel_size=1)\n        else:\n            groups = int(res_groups) if res_groups is not None else 1\n            if channels % groups != 0:\n                raise ValueError(f\"channels={channels} is not divisible by res_groups={groups}\")\n            self.conv = nn.Conv2d(channels, channels, kernel_size=(4, 3), groups=groups)\n\n        self.bn = nn.BatchNorm2d(channels)\n        self.elu = ActivationFactory(prelu_type, channels)\n        self.eca_f = _attention(channels, use_eca_f)\n\n    def forward(self, x):\n        \"\"\"x: (B, C, T, F).\"\"\"\n        if self.dw_residual:\n            y = self.pointwise(self.depthwise(self.pad(x)))\n        else:\n            y = self.conv(self.pad(x))\n        y = self.eca_f(self.elu(self.bn(y)))\n        return y + x\n\n\nclass EncoderBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        in_channels,\n        out_channels,\n        kernel_size=(4, 3),\n        stride=(1, 2),\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        res_groups=None,\n    ):\n        super().__init__()\n        self.pad = nn.ZeroPad2d([1, 1, 3, 0])\n        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride)\n        self.bn = nn.BatchNorm2d(out_channels)\n        self.elu = ActivationFactory(prelu_type, out_channels)\n        self.main_eca_f = _attention(out_channels, main_block_eca_f)\n        self.resblock = ResidualBlock_Ablation(\n            out_channels,\n            prelu_type=prelu_type,\n            dw_residual=dw_residual,\n            use_eca_f=use_eca_f,\n            res_groups=res_groups,\n        )\n\n    def forward(self, x):\n        y = self.elu(self.bn(self.conv(self.pad(x))))\n        y = self.main_eca_f(y)\n        return self.resblock(y)\n\n\nclass Bottleneck_Ablation(nn.Module):\n    def __init__(self, input_size, hidden_size, temporal_refine=False, temporal_refine_kernel=3):\n        super().__init__()\n        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)\n        self.fc = nn.Linear(hidden_size, input_size)\n        self.refine = (\n            CausalTemporalRefine(input_size, temporal_refine_kernel)\n            if temporal_refine\n            else None\n        )\n\n    def forward(self, x):\n        \"\"\"x: (B, C, T, F).\"\"\"\n        y = rearrange(x, \"b c t f -> b t (c f)\")\n        y = self.gru(y)[0]\n        y = self.fc(y)\n        if self.refine is not None:\n            y = y + self.refine(y)\n        y = rearrange(y, \"b t (c f) -> b c t f\", c=x.shape[1])\n        return y\n\n\nclass MambaBottleneck_Ablation(nn.Module):\n    def __init__(\n        self,\n        input_size,\n        num_blocks=2,\n        hidden_size=None,\n        d_state=16,\n        d_conv=4,\n        expand=2,\n    ):\n        super().__init__()\n        self.input_size = int(input_size)\n        self.stack = MambaStack(\n            self.input_size,\n            num_blocks=num_blocks,\n            hidden_size=hidden_size,\n            d_state=d_state,\n            d_conv=d_conv,\n            expand=expand,\n        )\n\n    def forward(self, x):\n        \"\"\"x: (B, C, T, F).\"\"\"\n        y = rearrange(x, \"b c t f -> b t (c f)\")\n        y, _ = self.stack(y)\n        y = rearrange(y, \"b t (c f) -> b c t f\", c=x.shape[1])\n        return y\n\n\nclass DecoderBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        in_channels,\n        out_channels,\n        kernel_size=(4, 3),\n        is_last=False,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        res_groups=None,\n        skip_gate=None,\n        dw_subpixel=False,\n    ):\n        super().__init__()\n        self.skip_conv = nn.Conv2d(in_channels, in_channels, 1)\n        self.skip_gate = _skip_gate(in_channels, skip_gate)\n        self.resblock = ResidualBlock_Ablation(\n            in_channels,\n            prelu_type=prelu_type,\n            dw_residual=dw_residual,\n            use_eca_f=use_eca_f,\n            res_groups=res_groups,\n        )\n        if dw_subpixel:\n            self.deconv = DWSubpixelConv2d(in_channels, out_channels, kernel_size)\n        else:\n            self.deconv = SubpixelConv2d(in_channels, out_channels, kernel_size)\n        self.bn = nn.BatchNorm2d(out_channels)\n        self.elu = ActivationFactory(prelu_type, out_channels)\n        self.is_last = is_last\n        self.main_eca_f = _attention(out_channels, main_block_eca_f and not is_last)\n\n    def forward(self, x, x_en):\n        y = x + self.skip_gate(self.skip_conv(x_en))\n        y = self.deconv(self.resblock(y))\n        if not self.is_last:\n            y = self.elu(self.bn(y))\n            y = self.main_eca_f(y)\n        return y\n\n\nclass DeepVQE_Ablation(nn.Module):\n    def __init__(\n        self,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        gru_hidden=BASE_GRU_HIDDEN,\n        skip_gate=None,\n        dw_subpixel=False,\n        temporal_refine=False,\n        temporal_refine_kernel=3,\n        sequence_model=\"gru\",\n        mamba_blocks=2,\n        mamba_hidden=None,\n        mamba_state=16,\n        mamba_conv=4,\n        mamba_expand=2,\n        **legacy_kwargs,\n    ):\n        super().__init__()\n        cfg = _normalize_model_config(\n            {\n                \"prelu_type\": prelu_type,\n                \"dw_residual\": dw_residual,\n                \"use_eca_f\": use_eca_f,\n                \"main_block_eca_f\": main_block_eca_f,\n                \"gru_hidden\": gru_hidden,\n                \"skip_gate\": skip_gate,\n                \"dw_subpixel\": dw_subpixel,\n                \"temporal_refine\": temporal_refine,\n                \"temporal_refine_kernel\": temporal_refine_kernel,\n                \"sequence_model\": sequence_model,\n                \"mamba_blocks\": mamba_blocks,\n                \"mamba_hidden\": mamba_hidden,\n                \"mamba_state\": mamba_state,\n                \"mamba_conv\": mamba_conv,\n                \"mamba_expand\": mamba_expand,\n                **legacy_kwargs,\n            }\n        )\n        self.ablation_config = deepcopy(cfg)\n\n        block_kwargs = {\n            \"prelu_type\": cfg[\"prelu_type\"],\n            \"dw_residual\": cfg[\"dw_residual\"],\n            \"use_eca_f\": cfg[\"use_eca_f\"],\n            \"main_block_eca_f\": cfg[\"main_block_eca_f\"],\n            \"res_groups\": cfg[\"res_groups\"],\n        }\n        decoder_kwargs = {\n            **block_kwargs,\n            \"skip_gate\": cfg[\"skip_gate\"],\n            \"dw_subpixel\": cfg[\"dw_subpixel\"],\n        }\n\n        self.fe = FE()\n        self.enblock1 = EncoderBlock_Ablation(2, 64, **block_kwargs)\n        self.enblock2 = EncoderBlock_Ablation(64, 128, **block_kwargs)\n        self.enblock3 = EncoderBlock_Ablation(128, 128, **block_kwargs)\n        self.enblock4 = EncoderBlock_Ablation(128, 128, **block_kwargs)\n        self.enblock5 = EncoderBlock_Ablation(128, 128, **block_kwargs)\n\n        if cfg[\"sequence_model\"] == \"mamba\":\n            self.bottle = MambaBottleneck_Ablation(\n                128 * 9,\n                num_blocks=cfg[\"mamba_blocks\"],\n                hidden_size=cfg[\"mamba_hidden\"],\n                d_state=cfg[\"mamba_state\"],\n                d_conv=cfg[\"mamba_conv\"],\n                expand=cfg[\"mamba_expand\"],\n            )\n        else:\n            self.bottle = Bottleneck_Ablation(\n                128 * 9,\n                int(cfg[\"gru_hidden\"]),\n                temporal_refine=cfg[\"temporal_refine\"],\n                temporal_refine_kernel=cfg[\"temporal_refine_kernel\"],\n            )\n\n        self.deblock5 = DecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock4 = DecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock3 = DecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock2 = DecoderBlock_Ablation(128, 64, **decoder_kwargs)\n        # Keep the original final activation for Baseline parity, but never\n        # attach main-block ECA-F to the output mask branch.\n        last_kwargs = dict(decoder_kwargs)\n        last_kwargs[\"main_block_eca_f\"] = False\n        self.deblock1 = DecoderBlock_Ablation(64, 27, **last_kwargs)\n        self.ccm = CCM()\n\n    @classmethod\n    def from_config_id(cls, config_id, **overrides):\n        return cls(**get_ablation_config(config_id, **overrides))\n\n    def forward(self, x):\n        \"\"\"x: (B, F, T, 2).\"\"\"\n        en_x0 = self.fe(x)\n        en_x1 = self.enblock1(en_x0)\n        en_x2 = self.enblock2(en_x1)\n        en_x3 = self.enblock3(en_x2)\n        en_x4 = self.enblock4(en_x3)\n        en_x5 = self.enblock5(en_x4)\n\n        en_xr = self.bottle(en_x5)\n\n        de_x5 = self.deblock5(en_xr, en_x5)[..., : en_x4.shape[-1]]\n        de_x4 = self.deblock4(de_x5, en_x4)[..., : en_x3.shape[-1]]\n        de_x3 = self.deblock3(de_x4, en_x3)[..., : en_x2.shape[-1]]\n        de_x2 = self.deblock2(de_x3, en_x2)[..., : en_x1.shape[-1]]\n        de_x1 = self.deblock1(de_x2, en_x1)[..., : en_x0.shape[-1]]\n\n        return self.ccm(de_x1, x)\n\n\nclass StreamResidualBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        channels,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        res_groups=None,\n    ):\n        super().__init__()\n        channels = int(channels)\n        self.dw_residual = bool(dw_residual)\n        self.res_groups = res_groups\n\n        if self.dw_residual:\n            self.depthwise = StreamConv2d(\n                channels,\n                channels,\n                kernel_size=(4, 3),\n                padding=(0, 1),\n                groups=channels,\n            )\n            self.pointwise = nn.Conv2d(channels, channels, kernel_size=1)\n        else:\n            groups = int(res_groups) if res_groups is not None else 1\n            if channels % groups != 0:\n                raise ValueError(f\"channels={channels} is not divisible by res_groups={groups}\")\n            self.conv = StreamConv2d(\n                channels,\n                channels,\n                kernel_size=(4, 3),\n                padding=(0, 1),\n                groups=groups,\n            )\n\n        self.bn = nn.BatchNorm2d(channels)\n        self.elu = ActivationFactory(prelu_type, channels)\n        self.eca_f = _attention(channels, use_eca_f)\n\n    def forward(self, x, cache):\n        \"\"\"x: (B, C, 1, F), cache: (B, C, 3, F).\"\"\"\n        if self.dw_residual:\n            y, cache = self.depthwise(x, cache)\n            y = self.pointwise(y)\n        else:\n            y, cache = self.conv(x, cache)\n        y = self.eca_f(self.elu(self.bn(y)))\n        return y + x, cache\n\n\nclass StreamEncoderBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        in_channels,\n        out_channels,\n        kernel_size=(4, 3),\n        stride=(1, 2),\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        res_groups=None,\n    ):\n        super().__init__()\n        self.conv = StreamConv2d(\n            in_channels,\n            out_channels,\n            kernel_size,\n            stride=stride,\n            padding=(0, 1),\n        )\n        self.bn = nn.BatchNorm2d(out_channels)\n        self.elu = ActivationFactory(prelu_type, out_channels)\n        self.main_eca_f = _attention(out_channels, main_block_eca_f)\n        self.resblock = StreamResidualBlock_Ablation(\n            out_channels,\n            prelu_type=prelu_type,\n            dw_residual=dw_residual,\n            use_eca_f=use_eca_f,\n            res_groups=res_groups,\n        )\n\n    def forward(self, x, conv_cache, res_cache):\n        x, conv_cache = self.conv(x, conv_cache)\n        x = self.main_eca_f(self.elu(self.bn(x)))\n        x, res_cache = self.resblock(x, res_cache)\n        return x, conv_cache, res_cache\n\n\nclass StreamBottleneck_Ablation(nn.Module):\n    def __init__(self, input_size, hidden_size, temporal_refine=False, temporal_refine_kernel=3):\n        super().__init__()\n        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)\n        self.fc = nn.Linear(hidden_size, input_size)\n        self.refine = (\n            CausalTemporalRefine(input_size, temporal_refine_kernel)\n            if temporal_refine\n            else None\n        )\n        self.refine_cache_size = int(temporal_refine_kernel) - 1 if temporal_refine else 0\n\n    def forward(self, x, cache, refine_cache=None):\n        \"\"\"x: (B, C, 1, F), cache: (1, B, hidden_size).\"\"\"\n        y = rearrange(x, \"b c t f -> b t (c f)\")\n        y, cache = self.gru(y, cache)\n        y = self.fc(y)\n        if self.refine is not None:\n            if refine_cache is None:\n                raise ValueError(\"refine_cache is required when temporal_refine=True\")\n            if refine_cache.shape[2] != self.refine_cache_size:\n                raise ValueError(\n                    f\"Expected refine_cache length {self.refine_cache_size}, got {refine_cache.shape[2]}\"\n                )\n            z = self.refine.norm(y).transpose(1, 2)\n            if self.refine_cache_size > 0:\n                z_with_history = torch.cat([refine_cache, z], dim=2)\n                refine_cache = z_with_history[:, :, -self.refine_cache_size :].contiguous()\n            else:\n                z_with_history = z\n            z = self.refine.depthwise(z_with_history)\n            z = self.refine.pointwise(z)\n            z = self.refine.elu(z)\n            y = y + z.transpose(1, 2)\n        y = rearrange(y, \"b t (c f) -> b c t f\", f=x.shape[-1])\n        return y, cache, refine_cache\n\n\nclass StreamMambaBottleneck_Ablation(nn.Module):\n    def __init__(\n        self,\n        input_size,\n        num_blocks=2,\n        hidden_size=None,\n        d_state=16,\n        d_conv=4,\n        expand=2,\n    ):\n        super().__init__()\n        self.input_size = int(input_size)\n        self.refine = None\n        self.stack = MambaStack(\n            self.input_size,\n            num_blocks=num_blocks,\n            hidden_size=hidden_size,\n            d_state=d_state,\n            d_conv=d_conv,\n            expand=expand,\n        )\n\n    def cache_names(self):\n        return self.stack.cache_names(\"mamba\")\n\n    def init_cache(self, batch_size, device=None, dtype=None):\n        return list(self.stack.init_cache(batch_size, device=device, dtype=dtype))\n\n    def forward(self, x, cache, refine_cache=None):\n        \"\"\"x: (B, C, 1, F), cache: flat Mamba cache list.\"\"\"\n        y = rearrange(x, \"b c t f -> b t (c f)\")\n        if y.shape[1] != 1:\n            raise ValueError(\"StreamMambaBottleneck_Ablation expects one frame at a time\")\n        y, cache = self.stack.step(y[:, 0], tuple(cache))\n        y = rearrange(y.unsqueeze(1), \"b t (c f) -> b c t f\", f=x.shape[-1])\n        return y, list(cache), refine_cache\n\n\nclass StreamSubpixelConv2d_Ablation(nn.Module):\n    def __init__(self, in_channels, out_channels, kernel_size=(4, 3)):\n        super().__init__()\n        self.conv = StreamConv2d(in_channels, out_channels * 2, kernel_size, padding=(0, 1))\n\n    def forward(self, x, cache):\n        \"\"\"x: (B, C, 1, F), cache: (B, C, 3, F).\"\"\"\n        y, cache = self.conv(x, cache)\n        y = rearrange(y, \"b (r c) t f -> b c t (r f)\", r=2)\n        return y, cache\n\n\nclass StreamDWSubpixelConv2d_Ablation(nn.Module):\n    def __init__(self, in_channels, out_channels, kernel_size=(4, 3)):\n        super().__init__()\n        self.depthwise = StreamConv2d(\n            in_channels,\n            in_channels,\n            kernel_size,\n            padding=(0, 1),\n            groups=in_channels,\n        )\n        self.pointwise = nn.Conv2d(in_channels, out_channels * 2, kernel_size=1)\n\n    def forward(self, x, cache):\n        \"\"\"x: (B, C, 1, F), cache: (B, C, 3, F).\"\"\"\n        y, cache = self.depthwise(x, cache)\n        y = self.pointwise(y)\n        y = rearrange(y, \"b (r c) t f -> b c t (r f)\", r=2)\n        return y, cache\n\n\nclass StreamDecoderBlock_Ablation(nn.Module):\n    def __init__(\n        self,\n        in_channels,\n        out_channels,\n        kernel_size=(4, 3),\n        is_last=False,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        res_groups=None,\n        skip_gate=None,\n        dw_subpixel=False,\n    ):\n        super().__init__()\n        self.skip_conv = nn.Conv2d(in_channels, in_channels, 1)\n        self.skip_gate = _skip_gate(in_channels, skip_gate)\n        self.resblock = StreamResidualBlock_Ablation(\n            in_channels,\n            prelu_type=prelu_type,\n            dw_residual=dw_residual,\n            use_eca_f=use_eca_f,\n            res_groups=res_groups,\n        )\n        if dw_subpixel:\n            self.deconv = StreamDWSubpixelConv2d_Ablation(in_channels, out_channels, kernel_size)\n        else:\n            self.deconv = StreamSubpixelConv2d_Ablation(in_channels, out_channels, kernel_size)\n        self.bn = nn.BatchNorm2d(out_channels)\n        self.elu = ActivationFactory(prelu_type, out_channels)\n        self.is_last = is_last\n        self.main_eca_f = _attention(out_channels, main_block_eca_f and not is_last)\n\n    def forward(self, x, x_en, conv_cache, res_cache):\n        y = x + self.skip_gate(self.skip_conv(x_en))\n        y, res_cache = self.resblock(y, res_cache)\n        y, conv_cache = self.deconv(y, conv_cache)\n        if not self.is_last:\n            y = self.elu(self.bn(y))\n            y = self.main_eca_f(y)\n        return y, conv_cache, res_cache\n\n\nclass StreamCCM_Ablation(nn.Module):\n    \"\"\"Stateful Complex Convolving Mask block.\"\"\"\n\n    def __init__(self):\n        super().__init__()\n        self.v = torch.tensor(\n            [[1.0, -0.5, -0.5], [0.0, 0.8660254037844386, -0.8660254037844386]],\n            dtype=torch.float32,\n        )\n        self.unfold = nn.Unfold(kernel_size=(3, 3), padding=(0, 1))\n\n    def forward(self, m, x, cache):\n        \"\"\"\n        m: (B, 27, 1, F)\n        x: (B, F, 1, 2)\n        cache: (B, F, 2, 2)\n        \"\"\"\n        m = rearrange(m, \"b (r c) t f -> b r c t f\", r=3)\n        v = self.v.to(device=m.device, dtype=m.dtype)\n        h_real = torch.sum(v[0][None, :, None, None, None] * m, dim=1)\n        h_imag = torch.sum(v[1][None, :, None, None, None] * m, dim=1)\n\n        m_real = rearrange(h_real, \"b (m n) t f -> b m n t f\", m=3)\n        m_imag = rearrange(h_imag, \"b (m n) t f -> b m n t f\", m=3)\n\n        x = torch.cat([cache, x], dim=2)\n        cache = x[:, :, 1:].contiguous()\n        x = x.permute(0, 3, 2, 1).contiguous()\n\n        x_unfold = self.unfold(x)\n        x_unfold = rearrange(\n            x_unfold,\n            \"b (c m n) (t f) -> b c m n t f\",\n            c=2,\n            m=3,\n            n=3,\n            f=x.shape[-1],\n        )\n\n        x_enh_real = torch.sum(m_real * x_unfold[:, 0] - m_imag * x_unfold[:, 1], dim=(1, 2))\n        x_enh_imag = torch.sum(m_real * x_unfold[:, 1] + m_imag * x_unfold[:, 0], dim=(1, 2))\n        x_enh = torch.stack([x_enh_real, x_enh_imag], dim=3).transpose(1, 2).contiguous()\n        return x_enh, cache\n\n\nclass StreamDeepVQE_Ablation(nn.Module):\n    \"\"\"Stateful streaming counterpart for every ``DeepVQE_Ablation`` variant.\"\"\"\n\n    cache_prefix_names = (\n        \"en_conv_cache1\",\n        \"en_res_cache1\",\n        \"en_conv_cache2\",\n        \"en_res_cache2\",\n        \"en_conv_cache3\",\n        \"en_res_cache3\",\n        \"en_conv_cache4\",\n        \"en_res_cache4\",\n        \"en_conv_cache5\",\n        \"en_res_cache5\",\n    )\n    cache_suffix_names = (\n        \"de_conv_cache5\",\n        \"de_res_cache5\",\n        \"de_conv_cache4\",\n        \"de_res_cache4\",\n        \"de_conv_cache3\",\n        \"de_res_cache3\",\n        \"de_conv_cache2\",\n        \"de_res_cache2\",\n        \"de_conv_cache1\",\n        \"de_res_cache1\",\n        \"m_cache\",\n    )\n\n    def get_cache_names(self):\n        bottle_names = (\n            self.bottle.cache_names()\n            if hasattr(self.bottle, \"cache_names\")\n            else (\"h_cache\",)\n        )\n        names = self.cache_prefix_names + tuple(bottle_names)\n        if getattr(self.bottle, \"refine\", None) is not None:\n            names = names + (\"temporal_refine_cache\",)\n        return names + self.cache_suffix_names\n\n    def __init__(\n        self,\n        prelu_type=None,\n        dw_residual=False,\n        use_eca_f=False,\n        main_block_eca_f=False,\n        gru_hidden=BASE_GRU_HIDDEN,\n        skip_gate=None,\n        dw_subpixel=False,\n        temporal_refine=False,\n        temporal_refine_kernel=3,\n        sequence_model=\"gru\",\n        mamba_blocks=2,\n        mamba_hidden=None,\n        mamba_state=16,\n        mamba_conv=4,\n        mamba_expand=2,\n        **legacy_kwargs,\n    ):\n        super().__init__()\n        cfg = _normalize_model_config(\n            {\n                \"prelu_type\": prelu_type,\n                \"dw_residual\": dw_residual,\n                \"use_eca_f\": use_eca_f,\n                \"main_block_eca_f\": main_block_eca_f,\n                \"gru_hidden\": gru_hidden,\n                \"skip_gate\": skip_gate,\n                \"dw_subpixel\": dw_subpixel,\n                \"temporal_refine\": temporal_refine,\n                \"temporal_refine_kernel\": temporal_refine_kernel,\n                \"sequence_model\": sequence_model,\n                \"mamba_blocks\": mamba_blocks,\n                \"mamba_hidden\": mamba_hidden,\n                \"mamba_state\": mamba_state,\n                \"mamba_conv\": mamba_conv,\n                \"mamba_expand\": mamba_expand,\n                **legacy_kwargs,\n            }\n        )\n        self.ablation_config = deepcopy(cfg)\n\n        block_kwargs = {\n            \"prelu_type\": cfg[\"prelu_type\"],\n            \"dw_residual\": cfg[\"dw_residual\"],\n            \"use_eca_f\": cfg[\"use_eca_f\"],\n            \"main_block_eca_f\": cfg[\"main_block_eca_f\"],\n            \"res_groups\": cfg[\"res_groups\"],\n        }\n        decoder_kwargs = {\n            **block_kwargs,\n            \"skip_gate\": cfg[\"skip_gate\"],\n            \"dw_subpixel\": cfg[\"dw_subpixel\"],\n        }\n\n        self.fe = FE()\n        self.enblock1 = StreamEncoderBlock_Ablation(2, 64, **block_kwargs)\n        self.enblock2 = StreamEncoderBlock_Ablation(64, 128, **block_kwargs)\n        self.enblock3 = StreamEncoderBlock_Ablation(128, 128, **block_kwargs)\n        self.enblock4 = StreamEncoderBlock_Ablation(128, 128, **block_kwargs)\n        self.enblock5 = StreamEncoderBlock_Ablation(128, 128, **block_kwargs)\n\n        if cfg[\"sequence_model\"] == \"mamba\":\n            self.bottle = StreamMambaBottleneck_Ablation(\n                128 * 9,\n                num_blocks=cfg[\"mamba_blocks\"],\n                hidden_size=cfg[\"mamba_hidden\"],\n                d_state=cfg[\"mamba_state\"],\n                d_conv=cfg[\"mamba_conv\"],\n                expand=cfg[\"mamba_expand\"],\n            )\n        else:\n            self.bottle = StreamBottleneck_Ablation(\n                128 * 9,\n                int(cfg[\"gru_hidden\"]),\n                temporal_refine=cfg[\"temporal_refine\"],\n                temporal_refine_kernel=cfg[\"temporal_refine_kernel\"],\n            )\n\n        self.deblock5 = StreamDecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock4 = StreamDecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock3 = StreamDecoderBlock_Ablation(128, 128, **decoder_kwargs)\n        self.deblock2 = StreamDecoderBlock_Ablation(128, 64, **decoder_kwargs)\n        last_kwargs = dict(decoder_kwargs)\n        last_kwargs[\"main_block_eca_f\"] = False\n        self.deblock1 = StreamDecoderBlock_Ablation(64, 27, **last_kwargs)\n        self.ccm = StreamCCM_Ablation()\n\n    @classmethod\n    def from_config_id(cls, config_id, **overrides):\n        return cls(**get_ablation_config(config_id, **overrides))\n\n    @classmethod\n    def from_offline(cls, model, strict=True):\n        stream_model = cls(**model.ablation_config)\n        convert_ablation_to_stream(stream_model, model, strict=strict)\n        stream_model.train(model.training)\n        return stream_model\n\n    def init_cache(self, batch_size=1, freq_bins=257, device=None, dtype=None):\n        \"\"\"Create zero caches in the fixed order expected by ``forward``.\"\"\"\n        param = next(self.parameters())\n        device = device if device is not None else param.device\n        dtype = dtype if dtype is not None else param.dtype\n        b = int(batch_size)\n        f0 = int(freq_bins)\n        f1 = (f0 + 1) // 2\n        f2 = (f1 + 1) // 2\n        f3 = (f2 + 1) // 2\n        f4 = (f3 + 1) // 2\n        f5 = (f4 + 1) // 2\n        if hasattr(self.bottle, \"init_cache\"):\n            bottle_cache = self.bottle.init_cache(b, device=device, dtype=dtype)\n        else:\n            hidden = self.bottle.gru.hidden_size\n            bottle_cache = [torch.zeros(1, b, hidden, device=device, dtype=dtype)]\n\n        cache = [\n            torch.zeros(b, 2, 3, f0, device=device, dtype=dtype),\n            torch.zeros(b, 64, 3, f1, device=device, dtype=dtype),\n            torch.zeros(b, 64, 3, f1, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f2, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f2, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f3, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f3, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f4, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f4, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f5, device=device, dtype=dtype),\n            *bottle_cache,\n            torch.zeros(b, 128, 3, f5, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f5, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f4, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f4, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f3, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f3, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f2, device=device, dtype=dtype),\n            torch.zeros(b, 128, 3, f2, device=device, dtype=dtype),\n            torch.zeros(b, 64, 3, f1, device=device, dtype=dtype),\n            torch.zeros(b, 64, 3, f1, device=device, dtype=dtype),\n            torch.zeros(b, f0, 2, 2, device=device, dtype=dtype),\n        ]\n        if self.bottle.refine is not None:\n            refine_cache = torch.zeros(\n                b,\n                self.bottle.refine.features,\n                self.bottle.refine_cache_size,\n                device=device,\n                dtype=dtype,\n            )\n            cache.insert(len(self.cache_prefix_names) + len(bottle_cache), refine_cache)\n        return cache\n\n    def forward(self, x, cache):\n        \"\"\"\n        x: (B, F, 1, 2)\n        cache: list of tensors following ``cache_names``.\n        \"\"\"\n        cache_names = self.get_cache_names()\n        if len(cache) != len(cache_names):\n            raise ValueError(f\"Expected {len(cache_names)} cache tensors, got {len(cache)}\")\n\n        prefix_len = len(self.cache_prefix_names)\n        bottle_len = len(self.bottle.cache_names()) if hasattr(self.bottle, \"cache_names\") else 1\n        (\n            en_conv_cache1,\n            en_res_cache1,\n            en_conv_cache2,\n            en_res_cache2,\n            en_conv_cache3,\n            en_res_cache3,\n            en_conv_cache4,\n            en_res_cache4,\n            en_conv_cache5,\n            en_res_cache5,\n        ) = cache[:prefix_len]\n        bottle_cache = list(cache[prefix_len : prefix_len + bottle_len])\n        idx = prefix_len + bottle_len\n        if self.bottle.refine is not None:\n            temporal_refine_cache = cache[idx]\n            idx += 1\n        else:\n            temporal_refine_cache = None\n        (\n            de_conv_cache5,\n            de_res_cache5,\n            de_conv_cache4,\n            de_res_cache4,\n            de_conv_cache3,\n            de_res_cache3,\n            de_conv_cache2,\n            de_res_cache2,\n            de_conv_cache1,\n            de_res_cache1,\n            m_cache,\n        ) = cache[idx:]\n\n        en_x0 = self.fe(x)\n        en_x1, en_conv_cache1, en_res_cache1 = self.enblock1(en_x0, en_conv_cache1, en_res_cache1)\n        en_x2, en_conv_cache2, en_res_cache2 = self.enblock2(en_x1, en_conv_cache2, en_res_cache2)\n        en_x3, en_conv_cache3, en_res_cache3 = self.enblock3(en_x2, en_conv_cache3, en_res_cache3)\n        en_x4, en_conv_cache4, en_res_cache4 = self.enblock4(en_x3, en_conv_cache4, en_res_cache4)\n        en_x5, en_conv_cache5, en_res_cache5 = self.enblock5(en_x4, en_conv_cache5, en_res_cache5)\n\n        bottle_arg = bottle_cache if hasattr(self.bottle, \"cache_names\") else bottle_cache[0]\n        en_xr, bottle_cache, temporal_refine_cache = self.bottle(en_x5, bottle_arg, temporal_refine_cache)\n        if not isinstance(bottle_cache, (list, tuple)):\n            bottle_cache = [bottle_cache]\n\n        de_x5, de_conv_cache5, de_res_cache5 = self.deblock5(en_xr, en_x5, de_conv_cache5, de_res_cache5)\n        de_x5 = de_x5[..., : en_x4.shape[-1]]\n        de_x4, de_conv_cache4, de_res_cache4 = self.deblock4(de_x5, en_x4, de_conv_cache4, de_res_cache4)\n        de_x4 = de_x4[..., : en_x3.shape[-1]]\n        de_x3, de_conv_cache3, de_res_cache3 = self.deblock3(de_x4, en_x3, de_conv_cache3, de_res_cache3)\n        de_x3 = de_x3[..., : en_x2.shape[-1]]\n        de_x2, de_conv_cache2, de_res_cache2 = self.deblock2(de_x3, en_x2, de_conv_cache2, de_res_cache2)\n        de_x2 = de_x2[..., : en_x1.shape[-1]]\n        de_x1, de_conv_cache1, de_res_cache1 = self.deblock1(de_x2, en_x1, de_conv_cache1, de_res_cache1)\n        de_x1 = de_x1[..., : en_x0.shape[-1]]\n\n        x_enh, m_cache = self.ccm(de_x1, x, m_cache)\n\n        new_cache = [\n            en_conv_cache1,\n            en_res_cache1,\n            en_conv_cache2,\n            en_res_cache2,\n            en_conv_cache3,\n            en_res_cache3,\n            en_conv_cache4,\n            en_res_cache4,\n            en_conv_cache5,\n            en_res_cache5,\n            *bottle_cache,\n        ]\n        if self.bottle.refine is not None:\n            new_cache.append(temporal_refine_cache)\n        new_cache.extend([\n            de_conv_cache5,\n            de_res_cache5,\n            de_conv_cache4,\n            de_res_cache4,\n            de_conv_cache3,\n            de_res_cache3,\n            de_conv_cache2,\n            de_res_cache2,\n            de_conv_cache1,\n            de_res_cache1,\n            m_cache,\n        ])\n        return x_enh, new_cache\n\n    def forward_flat(self, x, *cache):\n        \"\"\"ONNX-friendly wrapper: returns ``(enh, *new_cache)``.\"\"\"\n        y, new_cache = self.forward(x, list(cache))\n        return (y, *new_cache)\n\n\nclass StreamDeepVQE_AblationONNXWrapper(nn.Module):\n    def __init__(self, stream_model):\n        super().__init__()\n        self.stream_model = stream_model\n\n    def forward(self, x, *cache):\n        return self.stream_model.forward_flat(x, *cache)\n\n\ndef convert_ablation_to_stream(stream_model, model, strict=True):\n    \"\"\"Copy offline ablation weights into the matching stateful stream model.\"\"\"\n    state_dict = model.state_dict()\n    new_state_dict = stream_model.state_dict()\n    missing = []\n    shape_mismatch = []\n\n    for key, value in new_state_dict.items():\n        candidates = (\n            key,\n            key.replace(\"Conv2d.\", \"\"),\n            key.replace(\"conv.Conv2d.\", \"conv.\"),\n            key.replace(\"depthwise.Conv2d.\", \"depthwise.\"),\n        )\n        matched = None\n        for candidate in candidates:\n            if candidate in state_dict:\n                matched = candidate\n                break\n        if matched is None:\n            missing.append(key)\n            continue\n        if state_dict[matched].shape != value.shape:\n            shape_mismatch.append((key, matched, tuple(value.shape), tuple(state_dict[matched].shape)))\n            continue\n        new_state_dict[key] = state_dict[matched]\n\n    if strict and (missing or shape_mismatch):\n        details = []\n        if missing:\n            details.append(f\"missing={missing}\")\n        if shape_mismatch:\n            details.append(f\"shape_mismatch={shape_mismatch}\")\n        raise ValueError(\"Unable to convert ablation weights to stream: \" + \"; \".join(details))\n\n    stream_model.load_state_dict(new_state_dict, strict=False)\n    return stream_model\n\n\n@torch.no_grad()\ndef stream_sequence(stream_model, x, cache=None):\n    \"\"\"Run a full ``(B, F, T, 2)`` sequence through a stateful stream model.\"\"\"\n    if cache is None:\n        cache = stream_model.init_cache(x.shape[0], x.shape[1], x.device, x.dtype)\n    outputs = []\n    for frame_idx in range(x.shape[2]):\n        y, cache = stream_model(x[:, :, frame_idx : frame_idx + 1, :], cache)\n        outputs.append(y)\n    return torch.cat(outputs, dim=2), cache\n\n\ndef count_parameters(model):\n    return sum(param.numel() for param in model.parameters())\n",
  "ablation/ablation_config.py": "from copy import deepcopy\nimport hashlib\nimport json\nimport os\nimport platform\nimport subprocess\n\nBASE_TRAIN_CONFIG = {\n    \"experiment\": {\n        \"name\": \"deepvqe_ablation\",\n        \"config_id\": \"Baseline\",\n        \"seed\": 1234,\n        \"output_dir\": \"runs/ablation/Baseline\",\n        \"resume_from\": None,\n    },\n    \"model\": {\n        \"prelu_type\": None,\n        \"dw_residual\": False,\n        \"use_eca_f\": False,\n        \"main_block_eca_f\": False,\n        \"skip_gate\": None,\n        \"dw_subpixel\": False,\n        \"gru_hidden\": 576,\n        \"sequence_model\": \"gru\",\n        \"mamba_blocks\": 2,\n        \"mamba_hidden\": None,\n        \"mamba_state\": 16,\n        \"mamba_conv\": 4,\n        \"mamba_expand\": 2,\n    },\n    \"stft\": {\n        \"n_fft\": 512,\n        \"hop_length\": 256,\n        \"win_length\": 512,\n        \"window\": \"sqrt_hann\",\n    },\n    \"data\": {\n        \"sample_rate\": 16000,\n        \"clip_seconds\": 3.0,\n        \"num_workers\": 2,\n        \"pin_memory\": True,\n        \"train_manifest\": \"data/manifests/train.jsonl\",\n        \"valid_manifest\": \"data/manifests/valid.jsonl\",\n        \"test_manifest\": \"data/manifests/test.jsonl\",\n        \"augment\": True,\n        \"aug_gain_range_db\": [-6.0, 6.0],\n        \"aug_snr_remix_range\": [0.0, 20.0],\n        \"aug_prob\": 0.5,\n    },\n    \"optimizer\": {\n        \"lr\": 1e-3,\n        \"weight_decay\": 0.0,\n        \"betas\": [0.9, 0.999],\n        \"grad_clip_norm\": 5.0,\n    },\n    \"scheduler\": {\n        \"mode\": \"min\",\n        \"factor\": 0.5,\n        \"patience\": 5,\n        \"min_lr\": 1e-6,\n    },\n    \"logging\": {\n        \"use_wandb\": True,\n        \"wandb_project\": \"DeepVQE-Ablation\",\n        \"eval_pesq_every\": 5,\n        \"eval_pesq_samples\": 50,\n        \"log_audio_every\": 5,\n    },\n    \"training\": {\n        \"device\": \"cuda\",\n        \"batch_size\": 8,\n        \"epochs\": 100,\n        \"checkpoint_monitor\": \"loss\",\n        \"checkpoint_mode\": \"min\",\n        \"use_amp\": True,\n        \"use_gan\": False,\n        \"num_d_scales\": 1,\n    },\n    \"loss\": {\n        \"lamda_ri\": 30.0,\n        \"lamda_mag\": 70.0,\n        \"compress_factor\": 0.3,\n        \"lamda_adv\": 0.05,\n        \"lambda_fm\": 0.0,\n    },\n}\n\n\nTRAIN_CONFIG_PRESETS = {\n    \"GAN_Baseline\": {\n        \"base_config_id\": \"Baseline\",\n        \"training\": {\n            \"use_gan\": True,\n            \"num_d_scales\": 1,\n        },\n        \"loss\": {\n            \"lamda_adv\": 0.05,\n            \"lambda_fm\": 0.0,\n        },\n    },\n    \"GAN_MSD3\": {\n        \"base_config_id\": \"Baseline\",\n        \"training\": {\n            \"use_gan\": True,\n            \"num_d_scales\": 3,\n        },\n        \"loss\": {\n            \"lamda_adv\": 0.05,\n            \"lambda_fm\": 2.0,\n        },\n    },\n    \"GAN_D1b_gru768\": {\n        \"base_config_id\": \"D1b_gru768\",\n        \"training\": {\n            \"use_gan\": True,\n            \"num_d_scales\": 3,\n        },\n        \"loss\": {\n            \"lamda_adv\": 0.05,\n            \"lambda_fm\": 2.0,\n        },\n    },\n    \"GAN_Mamba_b2_h384\": {\n        \"base_config_id\": \"Mamba_b2_h384\",\n        \"training\": {\n            \"use_gan\": True,\n            \"num_d_scales\": 3,\n        },\n        \"loss\": {\n            \"lamda_adv\": 0.05,\n            \"lambda_fm\": 2.0,\n        },\n    },\n}\n\n\ndef deep_update(base, override):\n    result = deepcopy(base)\n    for key, value in override.items():\n        if isinstance(value, dict) and isinstance(result.get(key), dict):\n            result[key] = deep_update(result[key], value)\n        else:\n            result[key] = deepcopy(value)\n    return result\n\n\ndef get_model_config_id(config_id):\n    \"\"\"Return the architecture config used by a train config or preset.\"\"\"\n    preset = TRAIN_CONFIG_PRESETS.get(config_id)\n    if preset is None:\n        return config_id\n    return preset[\"base_config_id\"]\n\n\ndef get_train_config(config_id=\"Baseline\"):\n    from ablation.deepvqe_ablation import get_ablation_config\n\n    model_config_id = get_model_config_id(config_id)\n    cfg = deepcopy(BASE_TRAIN_CONFIG)\n    cfg[\"experiment\"][\"config_id\"] = config_id\n    cfg[\"experiment\"][\"name\"] = f\"deepvqe_ablation_{config_id}\"\n    cfg[\"experiment\"][\"output_dir\"] = f\"runs/ablation/{config_id}\"\n    cfg[\"model\"] = get_ablation_config(model_config_id)\n    preset = TRAIN_CONFIG_PRESETS.get(config_id)\n    if preset is not None:\n        preset_override = {key: value for key, value in preset.items() if key != \"base_config_id\"}\n        cfg = deep_update(cfg, preset_override)\n    return cfg\n\n\ndef stable_config_hash(cfg):\n    payload = json.dumps(cfg, sort_keys=True, separators=(\",\", \":\"), default=str)\n    return hashlib.sha256(payload.encode(\"utf-8\")).hexdigest()[:12]\n\n\ndef git_commit():\n    try:\n        return subprocess.check_output(\n            [\"git\", \"rev-parse\", \"HEAD\"],\n            text=True,\n            stderr=subprocess.DEVNULL,\n        ).strip()\n    except Exception:\n        return \"\"\n\n\ndef hardware_info():\n    try:\n        import torch\n    except ImportError:\n        return {\n            \"platform\": platform.platform(),\n            \"processor\": platform.processor(),\n            \"cpu_count\": os.cpu_count(),\n            \"cuda_available\": False,\n            \"cuda_device\": \"\",\n        }\n\n    cuda_name = \"\"\n    if torch.cuda.is_available():\n        try:\n            cuda_name = torch.cuda.get_device_name(0)\n        except Exception:\n            cuda_name = \"cuda\"\n    return {\n        \"platform\": platform.platform(),\n        \"processor\": platform.processor(),\n        \"cpu_count\": os.cpu_count(),\n        \"cuda_available\": torch.cuda.is_available(),\n        \"cuda_device\": cuda_name,\n    }\n\n\ndef reproducibility_metadata(cfg=None, checkpoint_id=\"\"):\n    try:\n        import torch\n        torch_version = torch.__version__\n        num_threads = torch.get_num_threads()\n    except ImportError:\n        torch_version = \"\"\n        num_threads = \"\"\n\n    try:\n        import onnxruntime\n\n        ort_version = onnxruntime.__version__\n    except Exception:\n        ort_version = \"\"\n\n    return {\n        \"git_commit\": git_commit(),\n        \"config_hash\": stable_config_hash(cfg) if cfg is not None else \"\",\n        \"seed\": cfg.get(\"experiment\", {}).get(\"seed\", \"\") if cfg is not None else \"\",\n        \"hardware_info\": json.dumps(hardware_info(), sort_keys=True),\n        \"torch_version\": torch_version,\n        \"onnxruntime_version\": ort_version,\n        \"num_threads\": num_threads,\n        \"checkpoint_id\": checkpoint_id,\n    }\n",
  "ablation/configs/Mamba_b2_h384.yaml": "experiment:\n  config_id: Mamba_b2_h384\n  output_dir: runs/ablation/Mamba_b2_h384\nmodel:\n  prelu_type:\n  dw_residual: false\n  use_eca_f: false\n  main_block_eca_f: false\n  skip_gate:\n  dw_subpixel: false\n  gru_hidden: 576\n  sequence_model: mamba\n  mamba_blocks: 2\n  mamba_hidden: 384\n  mamba_state: 16\n  mamba_conv: 4\n  mamba_expand: 2\n\n",
  "ablation/configs/GAN_Mamba_b2_h384.yaml": "experiment:\n  config_id: GAN_Mamba_b2_h384\n  output_dir: runs/ablation/GAN_Mamba_b2_h384\nmodel:\n  prelu_type:\n  dw_residual: false\n  use_eca_f: false\n  main_block_eca_f: false\n  skip_gate:\n  dw_subpixel: false\n  gru_hidden: 576\n  sequence_model: mamba\n  mamba_blocks: 2\n  mamba_hidden: 384\n  mamba_state: 16\n  mamba_conv: 4\n  mamba_expand: 2\ntraining:\n  use_gan: true\n  num_d_scales: 3\nloss:\n  lamda_adv: 0.05\n  lambda_fm: 2.0\n\n",
  "ablation/test_phase2_mamba.py": "\"\"\"Smoke tests for Phase 2 Mamba sequence modeling.\n\nRun:\n    python ablation/test_phase2_mamba.py\n\"\"\"\n\nimport argparse\nimport random\nimport sys\nfrom pathlib import Path\n\nif __package__ in (None, \"\"):\n    sys.path.insert(0, str(Path(__file__).resolve().parents[1]))\n\nimport numpy as np\nimport torch\n\nfrom ablation.deepvqe_ablation import (\n    DeepVQE_Ablation,\n    MambaBottleneck_Ablation,\n    StreamDeepVQE_Ablation,\n    StreamMambaBottleneck_Ablation,\n    convert_ablation_to_stream,\n    get_ablation_config,\n    stream_sequence,\n)\n\n\ndef seed_everything(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n\n\n@torch.no_grad()\ndef test_mamba_bottleneck_parity(atol):\n    offline = MambaBottleneck_Ablation(\n        input_size=24,\n        num_blocks=2,\n        hidden_size=32,\n        d_state=8,\n        d_conv=4,\n        expand=2,\n    ).eval()\n    stream = StreamMambaBottleneck_Ablation(\n        input_size=24,\n        num_blocks=2,\n        hidden_size=32,\n        d_state=8,\n        d_conv=4,\n        expand=2,\n    ).eval()\n    stream.load_state_dict(offline.state_dict(), strict=True)\n\n    x = torch.randn(2, 3, 7, 8)\n    y_offline = offline(x)\n    cache = stream.init_cache(x.shape[0], x.device, x.dtype)\n    y_stream = []\n    for frame_idx in range(x.shape[2]):\n        y, cache, _ = stream(x[:, :, frame_idx : frame_idx + 1, :], cache)\n        y_stream.append(y)\n    y_stream = torch.cat(y_stream, dim=2)\n\n    max_abs_error = (y_offline - y_stream).abs().max().item()\n    if y_offline.shape != y_stream.shape:\n        raise AssertionError(f\"Mamba bottleneck shape mismatch: {y_offline.shape} != {y_stream.shape}\")\n    if len(cache) != len(stream.cache_names()):\n        raise AssertionError(f\"Mamba cache length mismatch: {len(cache)} != {len(stream.cache_names())}\")\n    if max_abs_error > atol:\n        raise AssertionError(f\"Mamba bottleneck streaming parity failed: {max_abs_error} > {atol}\")\n    print(f\"Mamba bottleneck parity passed max_abs_error={max_abs_error:.6g}\")\n\n\n@torch.no_grad()\ndef test_deepvqe_mamba_integration(frames, atol):\n    cfg = get_ablation_config(\"Mamba_b2_h384\")\n    model = DeepVQE_Ablation(**cfg).eval()\n    stream = StreamDeepVQE_Ablation(**cfg).eval()\n    convert_ablation_to_stream(stream, model, strict=True)\n\n    x = torch.randn(1, 257, frames, 2)\n    y_offline = model(x)\n    y_stream, cache = stream_sequence(stream, x)\n\n    cache_names = stream.get_cache_names()\n    expected_mamba_names = (\n        \"mamba1_conv_cache\",\n        \"mamba1_ssm_state\",\n        \"mamba2_conv_cache\",\n        \"mamba2_ssm_state\",\n    )\n    for name in expected_mamba_names:\n        if name not in cache_names:\n            raise AssertionError(f\"Missing stream cache name: {name}\")\n    if len(cache) != len(cache_names):\n        raise AssertionError(f\"DeepVQE Mamba cache length mismatch: {len(cache)} != {len(cache_names)}\")\n    if y_offline.shape != y_stream.shape:\n        raise AssertionError(f\"DeepVQE Mamba shape mismatch: {y_offline.shape} != {y_stream.shape}\")\n\n    max_abs_error = (y_offline - y_stream).abs().max().item()\n    if max_abs_error > atol:\n        raise AssertionError(f\"DeepVQE Mamba streaming parity failed: {max_abs_error} > {atol}\")\n    print(f\"DeepVQE Mamba integration passed max_abs_error={max_abs_error:.6g}\")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=\"Verify Phase 2 Mamba sequence modeling\")\n    parser.add_argument(\"--frames\", type=int, default=3)\n    parser.add_argument(\"--seed\", type=int, default=2026)\n    parser.add_argument(\"--atol\", type=float, default=1e-5)\n    args = parser.parse_args()\n\n    seed_everything(args.seed)\n    test_mamba_bottleneck_parity(args.atol)\n    test_deepvqe_mamba_integration(args.frames, args.atol)\n\n\nif __name__ == \"__main__\":\n    main()\n\n",
  "ablation/scripts/benchmark_rtf.py": "\"\"\"RTF benchmark for offline and stateful streaming ablation models.\n\nExamples:\n    python ablation/scripts/benchmark_rtf.py --configs Baseline Mamba_b2_h384\n    python ablation/scripts/benchmark_rtf.py --devices cpu cuda --frames 63\n\"\"\"\n\nimport argparse\nimport csv\nimport os\nimport sys\nimport time\nimport tracemalloc\nfrom pathlib import Path\n\nif __package__ in (None, \"\"):\n    sys.path.insert(0, str(Path(__file__).resolve().parents[2]))\n\nimport torch\n\nfrom ablation.ablation_config import get_model_config_id, get_train_config, reproducibility_metadata\nfrom ablation.deepvqe_ablation import (\n    ABLATION_CONFIGS,\n    DeepVQE_Ablation,\n    StreamDeepVQE_Ablation,\n    convert_ablation_to_stream,\n    count_parameters,\n    stream_sequence,\n)\n\n\ndef process_rss_mb():\n    try:\n        import psutil\n\n        return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)\n    except Exception:\n        return \"\"\n\n\ndef cuda_peak_mb(device):\n    if device.type != \"cuda\":\n        return \"\"\n    return torch.cuda.max_memory_allocated(device) / (1024 ** 2)\n\n\ndef synchronize(device):\n    if device.type == \"cuda\":\n        torch.cuda.synchronize(device)\n\n\n@torch.no_grad()\ndef benchmark_forward(fn, warmup, repeats, device):\n    for _ in range(warmup):\n        fn()\n    synchronize(device)\n    if device.type == \"cuda\":\n        torch.cuda.reset_peak_memory_stats(device)\n    tracemalloc.start()\n    start = time.perf_counter()\n    for _ in range(repeats):\n        fn()\n    synchronize(device)\n    elapsed = (time.perf_counter() - start) / max(1, repeats)\n    _, peak_py = tracemalloc.get_traced_memory()\n    tracemalloc.stop()\n    return elapsed, peak_py / (1024 ** 2), process_rss_mb(), cuda_peak_mb(device)\n\n\ndef available_devices(requested):\n    devices = []\n    for name in requested:\n        if name == \"cuda\" and not torch.cuda.is_available():\n            continue\n        devices.append(torch.device(name))\n    return devices\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=\"Benchmark DeepVQE ablation RTF\")\n    parser.add_argument(\"--configs\", nargs=\"*\", default=[\"Baseline\", \"D1b_gru768\", \"Mamba_b2_h384\"])\n    parser.add_argument(\"--devices\", nargs=\"*\", default=[\"cpu\", \"cuda\"])\n    parser.add_argument(\"--frames\", type=int, default=63)\n    parser.add_argument(\"--freq-bins\", type=int, default=257)\n    parser.add_argument(\"--sample-rate\", type=int, default=16000)\n    parser.add_argument(\"--hop-length\", type=int, default=256)\n    parser.add_argument(\"--warmup\", type=int, default=1)\n    parser.add_argument(\"--repeats\", type=int, default=3)\n    parser.add_argument(\"--output\", default=\"results/phase2_rtf_benchmark.csv\")\n    args = parser.parse_args()\n\n    devices = available_devices(args.devices)\n    if not devices:\n        raise RuntimeError(\"No requested benchmark devices are available\")\n\n    audio_seconds = args.frames * args.hop_length / args.sample_rate\n    rows = []\n    for config_id in args.configs:\n        if config_id not in ABLATION_CONFIGS:\n            model_config_id = get_model_config_id(config_id)\n        else:\n            model_config_id = config_id\n\n        for device in devices:\n            x = torch.randn(1, args.freq_bins, args.frames, 2, device=device)\n            model = DeepVQE_Ablation.from_config_id(model_config_id).eval().to(device)\n            stream_model = StreamDeepVQE_Ablation.from_config_id(model_config_id).eval().to(device)\n            convert_ablation_to_stream(stream_model, model, strict=True)\n\n            offline_seconds, offline_py_mb, offline_rss_mb, offline_cuda_mb = benchmark_forward(\n                lambda: model(x),\n                args.warmup,\n                args.repeats,\n                device,\n            )\n            stream_seconds, stream_py_mb, stream_rss_mb, stream_cuda_mb = benchmark_forward(\n                lambda: stream_sequence(stream_model, x),\n                args.warmup,\n                args.repeats,\n                device,\n            )\n            cache = stream_model.init_cache(1, args.freq_bins, device=device, dtype=x.dtype)\n\n            row = {\n                \"config_id\": config_id,\n                \"architecture\": model_config_id,\n                \"device\": str(device),\n                \"frames\": args.frames,\n                \"freq_bins\": args.freq_bins,\n                \"params\": count_parameters(model),\n                \"offline_rtf\": offline_seconds / audio_seconds,\n                \"offline_ms\": offline_seconds * 1000.0,\n                \"stream_rtf\": stream_seconds / audio_seconds,\n                \"stream_ms\": stream_seconds * 1000.0,\n                \"cache_tensors\": len(cache),\n                \"cache_names\": \"|\".join(stream_model.get_cache_names()),\n                \"offline_python_peak_mb\": offline_py_mb,\n                \"stream_python_peak_mb\": stream_py_mb,\n                \"offline_rss_mb\": offline_rss_mb,\n                \"stream_rss_mb\": stream_rss_mb,\n                \"offline_cuda_peak_mb\": offline_cuda_mb,\n                \"stream_cuda_peak_mb\": stream_cuda_mb,\n            }\n            row.update(reproducibility_metadata(get_train_config(config_id)))\n            rows.append(row)\n            print(\n                f\"{config_id} [{device}]: params={row['params']} \"\n                f\"offline_rtf={row['offline_rtf']:.4f} stream_rtf={row['stream_rtf']:.4f} \"\n                f\"cache_tensors={row['cache_tensors']}\"\n            )\n\n    output_path = Path(args.output)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    fieldnames = [\n        \"config_id\",\n        \"architecture\",\n        \"device\",\n        \"git_commit\",\n        \"config_hash\",\n        \"seed\",\n        \"hardware_info\",\n        \"torch_version\",\n        \"onnxruntime_version\",\n        \"num_threads\",\n        \"checkpoint_id\",\n        \"frames\",\n        \"freq_bins\",\n        \"params\",\n        \"offline_rtf\",\n        \"offline_ms\",\n        \"stream_rtf\",\n        \"stream_ms\",\n        \"cache_tensors\",\n        \"cache_names\",\n        \"offline_python_peak_mb\",\n        \"stream_python_peak_mb\",\n        \"offline_rss_mb\",\n        \"stream_rss_mb\",\n        \"offline_cuda_peak_mb\",\n        \"stream_cuda_peak_mb\",\n    ]\n    with output_path.open(\"w\", newline=\"\", encoding=\"utf-8\") as f:\n        writer = csv.DictWriter(f, fieldnames=fieldnames)\n        writer.writeheader()\n        writer.writerows(rows)\n    print(f\"Wrote {output_path}\")\n\n\nif __name__ == \"__main__\":\n    main()\n\n"
}

for rel_path, content in PHASE2_FILES.items():
    target = Path(rel_path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding='utf-8')
    print(f'Wrote {target}')

importlib.invalidate_caches()
print('Phase 2 Mamba runtime patch applied.')


## 2.6 Kiểm tra cấu hình Mamba


In [ ]:
from ablation.deepvqe_ablation import get_ablation_config, DeepVQE_Ablation, StreamDeepVQE_Ablation, count_parameters
from ablation.ablation_config import get_train_config, get_model_config_id

print('Model config id for GAN preset:', get_model_config_id('GAN_Mamba_b2_h384'))
cfg = get_ablation_config('Mamba_b2_h384')
print(cfg)

model = DeepVQE_Ablation.from_config_id('Mamba_b2_h384')
stream_model = StreamDeepVQE_Ablation.from_config_id('Mamba_b2_h384')
print(f'Mamba params: {count_parameters(model):,}')
print('Stream cache names:')
for name in stream_model.get_cache_names():
    print(' ', name)


## 3. Tải bộ dữ liệu VoiceBank-DEMAND


In [ ]:
import os
import zipfile
import subprocess
from pathlib import Path

data_dir = WORK_DIR / 'data' / 'voicebank-demand'
data_dir.mkdir(parents=True, exist_ok=True)

datasets = {
    'clean_trainset_28spk_wav': 'https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_trainset_28spk_wav.zip',
    'noisy_trainset_28spk_wav': 'https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_trainset_28spk_wav.zip',
    'clean_testset_wav': 'https://datashare.ed.ac.uk/bitstream/handle/10283/2791/clean_testset_wav.zip',
    'noisy_testset_wav': 'https://datashare.ed.ac.uk/bitstream/handle/10283/2791/noisy_testset_wav.zip',
}

for folder_name, url in datasets.items():
    extract_path = data_dir / folder_name
    zip_path = data_dir / f'{folder_name}.zip'
    if extract_path.exists() and any(extract_path.iterdir()):
        print(f'{folder_name}: đã có dữ liệu, bỏ qua.')
        continue
    if not zip_path.exists():
        print(f'Tải {folder_name}.zip...')
        subprocess.run(['wget', '-nc', '-P', str(data_dir), url], check=True)
    print(f'Giải nén {zip_path.name}...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(data_dir)

print(f'Dataset dir: {data_dir}')


## 4. Tạo CSV train/valid/test


In [ ]:
import glob
import pandas as pd

def create_csv(clean_dir, noisy_dir, output_csv):
    clean_files = sorted(glob.glob(str(Path(clean_dir) / '*.wav')))
    noisy_files = sorted(glob.glob(str(Path(noisy_dir) / '*.wav')))
    if len(clean_files) != len(noisy_files):
        raise ValueError(f'Số lượng file không khớp: {len(clean_files)} clean vs {len(noisy_files)} noisy')
    rows = []
    for clean_path, noisy_path in zip(clean_files, noisy_files):
        name = Path(clean_path).name
        rows.append({
            'ID': Path(name).stem,
            'clean_wav': str(Path(clean_path).resolve()),
            'noisy_wav': str(Path(noisy_path).resolve()),
        })
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f'Wrote {output_csv}: {len(df)} samples')

create_csv(data_dir / 'clean_trainset_28spk_wav', data_dir / 'noisy_trainset_28spk_wav', data_dir / 'train_full.csv')
create_csv(data_dir / 'clean_testset_wav', data_dir / 'noisy_testset_wav', data_dir / 'test.csv')

df_train_full = pd.read_csv(data_dir / 'train_full.csv').sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(df_train_full) * 0.90)
df_train_full.iloc[:split_idx].to_csv(data_dir / 'train.csv', index=False)
df_train_full.iloc[split_idx:].to_csv(data_dir / 'valid.csv', index=False)

print('train/valid/test ready:')
for name in ['train.csv', 'valid.csv', 'test.csv']:
    p = data_dir / name
    print(f'  {p}: {len(pd.read_csv(p))} rows')


## 5. Cấu hình chạy Phase 2


In [ ]:
import json
from pathlib import Path

CONFIG = {
    # Đổi thành 'GAN_Mamba_b2_h384' nếu muốn train Mamba cùng Phase 1 GAN loss.
    'config_id': 'Mamba_b2_h384',
    'run_name': 'phase2_mamba_b2_h384_v1',
    'seed': 1234,

    'sample_rate': 16000,
    'n_fft': 512,
    'hop_length': 256,
    'win_length': 512,
    'stft_window': 'sqrt_hann',

    'batch_size': 8,
    'epochs': 80,
    'num_workers': 2,
    'lr': 1e-3,
    'weight_decay': 0.0,
    'grad_clip_norm': 5.0,
    'early_stopping_patience': 15,

    'train_csv': str(data_dir / 'train.csv'),
    'valid_csv': str(data_dir / 'valid.csv'),
    'test_csv': str(data_dir / 'test.csv'),

    'output_dir': str(SHARED_CHECKPOINTS_DIR / 'phase2_mamba_b2_h384_v1'),
    'results_dir': str(WORK_DIR / 'results' / 'phase2_mamba_b2_h384_v1'),
    'onnx_dir': str(WORK_DIR / 'onnx_models' / 'phase2_mamba_b2_h384_v1'),
    'resume_existing': False,
    'resume_checkpoint': 'last.pt',

    'use_amp': True,
    'augment': True,
    'aug_gain_range_db': [-6.0, 6.0],
    'aug_snr_remix_range': [0.0, 20.0],
    'aug_prob': 0.5,

    'run_training': True,
    'run_smoke_tests': True,
    'run_rtf_benchmark': True,
    'run_quality_eval': True,
    'run_onnx_export': False,
}

Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['results_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['onnx_dir']).mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2))


## 6. Ghi config override cho `train_ablation.py`


In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
override = {
    'experiment': {
        'name': CONFIG['run_name'],
        'config_id': CONFIG['config_id'],
        'seed': CONFIG['seed'],
        'output_dir': CONFIG['output_dir'],
        'resume_from': str(Path(CONFIG['output_dir']) / CONFIG['resume_checkpoint']) if CONFIG['resume_existing'] else None,
    },
    'stft': {
        'n_fft': CONFIG['n_fft'],
        'hop_length': CONFIG['hop_length'],
        'win_length': CONFIG['win_length'],
        'window': CONFIG['stft_window'],
    },
    'data': {
        'sample_rate': CONFIG['sample_rate'],
        'clip_seconds': 3.0,
        'num_workers': CONFIG['num_workers'],
        'pin_memory': torch.cuda.is_available(),
        'train_manifest': CONFIG['train_csv'],
        'valid_manifest': CONFIG['valid_csv'],
        'test_manifest': CONFIG['test_csv'],
        'augment': CONFIG['augment'],
        'aug_gain_range_db': CONFIG['aug_gain_range_db'],
        'aug_snr_remix_range': CONFIG['aug_snr_remix_range'],
        'aug_prob': CONFIG['aug_prob'],
    },
    'optimizer': {
        'lr': CONFIG['lr'],
        'weight_decay': CONFIG['weight_decay'],
        'grad_clip_norm': CONFIG['grad_clip_norm'],
    },
    'training': {
        'device': device,
        'batch_size': CONFIG['batch_size'],
        'epochs': CONFIG['epochs'],
        'use_amp': CONFIG['use_amp'],
        'disable_tqdm': False,
    },
}

override_path = Path(CONFIG['output_dir']) / 'phase2_mamba_train_override.json'
override_path.write_text(json.dumps(override, indent=2), encoding='utf-8')
print(f'Override config: {override_path}')
print(f'Device: {device}')


## 7. Smoke test Phase 2 trước khi train


In [ ]:
import subprocess
import sys
import pandas as pd
from IPython.display import display, FileLink

def run_py(args, check=True):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(REPO_DIR), text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f'Lệnh lỗi với exit code {result.returncode}: {cmd}')
    return result

if CONFIG['run_smoke_tests']:
    run_py(['-m', 'py_compile', 'ablation/modules/mamba.py', 'ablation/deepvqe_ablation.py', 'ablation/test_phase2_mamba.py'])
    run_py(['ablation/test_phase2_mamba.py', '--frames', '3'])
    run_py(['ablation/test_ablation_streaming.py', '--configs', 'Mamba_b2_h384', '--frames', '3', '--atol', '1e-5'])
else:
    print('run_smoke_tests=False, bỏ qua smoke test.')


## 8. Train Mamba


In [ ]:
train_cmd = [
    'ablation/train_ablation.py',
    '--config-id', CONFIG['config_id'],
    '--config-json', override_path,
    '--train-manifest', CONFIG['train_csv'],
    '--valid-manifest', CONFIG['valid_csv'],
    '--output-dir', CONFIG['output_dir'],
    '--device', device,
    '--epochs', str(CONFIG['epochs']),
    '--batch-size', str(CONFIG['batch_size']),
    '--num-workers', str(CONFIG['num_workers']),
    '--early-stop-patience', str(CONFIG['early_stopping_patience']),
]
if CONFIG['resume_existing']:
    resume_path = Path(CONFIG['output_dir']) / CONFIG['resume_checkpoint']
    if resume_path.exists():
        train_cmd += ['--resume', resume_path]
if not torch.cuda.is_available():
    train_cmd += ['--no-pin-memory']

if CONFIG['run_training']:
    run_py(train_cmd)
else:
    print('run_training=False, bỏ qua training.')


## 9. Kiểm tra checkpoint & log


In [ ]:
output_dir = Path(CONFIG['output_dir'])
for name in ['best.pt', 'last.pt', 'config.json', 'train_log.txt']:
    p = output_dir / name
    if p.exists():
        print(f'{name}: {p} ({p.stat().st_size / 1024:.1f} KB)')
    else:
        print(f'MISSING: {p}')

log_path = output_dir / 'train_log.txt'
if log_path.exists():
    print('\nLast log lines:')
    print('\n'.join(log_path.read_text(encoding='utf-8', errors='ignore').splitlines()[-10:]))


## 10. RTF benchmark CPU/GPU


In [ ]:
rtf_csv = Path(CONFIG['results_dir']) / 'phase2_rtf_benchmark.csv'

if CONFIG['run_rtf_benchmark']:
    run_py([
        'ablation/scripts/benchmark_rtf.py',
        '--output', rtf_csv,
        '--configs', 'Baseline', 'D1b_gru768', 'Mamba_b2_h384',
        '--devices', 'cpu', 'cuda',
        '--frames', '63',
        '--warmup', '1',
        '--repeats', '3',
    ])
else:
    print('run_rtf_benchmark=False, bỏ qua RTF benchmark.')

if rtf_csv.exists():
    display(pd.read_csv(rtf_csv))


## 11. Quality eval trên test set


In [ ]:
quality_csv = Path(CONFIG['results_dir']) / 'ablation_quality.csv'
best_ckpt = Path(CONFIG['output_dir']) / 'best.pt'

if CONFIG['run_quality_eval']:
    if not best_ckpt.exists():
        raise FileNotFoundError(f'Không tìm thấy best checkpoint: {best_ckpt}')
    run_py([
        'ablation/eval_ablation_quality.py',
        '--config-id', CONFIG['config_id'],
        '--checkpoint', best_ckpt,
        '--manifest', CONFIG['test_csv'],
        '--output', quality_csv,
        '--device', device,
    ])
else:
    print('run_quality_eval=False, bỏ qua quality eval.')

if quality_csv.exists():
    display(pd.read_csv(quality_csv))


## 12. ONNX export streaming


In [ ]:
onnx_csv = Path(CONFIG['results_dir']) / 'ablation_onnx.csv'

if CONFIG['run_onnx_export']:
    if not best_ckpt.exists():
        raise FileNotFoundError(f'Không tìm thấy best checkpoint: {best_ckpt}')
    run_py([
        'ablation/export_ablation_onnx.py',
        '--config-id', CONFIG['config_id'],
        '--checkpoint', best_ckpt,
        '--output-dir', CONFIG['onnx_dir'],
        '--results', onnx_csv,
        '--device', 'cpu',
    ])
else:
    print('run_onnx_export=False, bỏ qua ONNX export.')

if onnx_csv.exists():
    display(pd.read_csv(onnx_csv))


## 13. Nén kết quả để tải về


In [ ]:
archive_base = Path('/kaggle/working/phase2_mamba_training_output')
archive_path = archive_base.with_suffix('.zip')
if archive_path.exists():
    archive_path.unlink()

shutil.make_archive(str(archive_base), 'zip', root_dir=str(WORK_DIR))
print(f'Đã nén kết quả: {archive_path}')
display(FileLink(str(archive_path)))
